# [M2] DẠY MÁY HỌC TỪ DỮ LIỆU
## Buổi S05: Huấn Luyện Model Đầu Tiên & Báo Cáo Kết Quả Chuẩn Thi Đấu

---

### Metadata Khóa Học
* **Module:** M2 — DẠY MÁY HỌC TỪ DỮ LIỆU
* **Buổi:** S05
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Model phân loại/dự đoán đầu tiên & Báo cáo kết quả (ML Result Report)
* **Mức độ liên quan VOAI / Olympic AI:** Hoàn thiện năng lực huấn luyện và báo cáo thử nghiệm mô hình theo chuẩn thi đấu.

#### Mục tiêu học tập (Learning Objectives)
- Xây dựng một End-to-End Machine Learning Pipeline khép kín: Tiền xử lý -> Chuẩn hóa đặc trưng -> Huấn luyện -> Đánh giá đa metric.
- So sánh định lượng hiệu năng giữa 3 cấp độ mô hình: Baseline ngây thơ vs Mô hình tuyến tính vs Mô hình cây (Decision Tree / Random Forest).
- Thực hiện phân tích lỗi sai (Error Analysis) để tìm ra điểm mù của mô hình.
- Hoàn thành sản phẩm đầu ra chính thức của Module 2: Huấn luyện model đầu tiên và xuất Báo cáo kết quả chuẩn thi đấu (ML Result Report).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành trọn vẹn 4 buổi trước (M2_S01 đến M2_S04).
- Nắm vững các bước làm sạch dữ liệu từ Module M1.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Chào mừng bạn đến với vạch đích của Module 2! Trong mọi cuộc thi AI, người chiến thắng không phải là người gọi được nhiều thư viện nhất, mà là người có một **Quy trình Thực nghiệm khoa học, chặt chẽ và có thể tái lập (Disciplined ML Workflow)**.

Hôm nay, bạn sẽ không làm các bài tập manh mún nữa. Bạn sẽ đóng vai một thí sinh thực thụ: Nhận một bài toán thực tế, dựng Baseline, chuẩn hóa dữ liệu, đưa 3 thuật toán vào "đấu trường so găng", phân tích số liệu trung thực bằng các metric đã học, và xuất bản một **Báo cáo Kết quả (ML Result Report)** hoàn chỉnh. Đây chính là cột mốc biến bạn từ một người học code thành một Kỹ sư Machine Learning thực thụ!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Chuẩn hóa đặc trưng (Feature Scaling)
Nhiều thuật toán nhạy cảm với thang đo của dữ liệu (ví dụ: Tuổi từ 18-60 nhưng Lương từ 10 triệu - 100 triệu). Nếu không chuẩn hóa, thuật toán tối ưu sẽ bị kéo lệch theo biến có biên độ lớn.
* **StandardScaler (Z-score Standardization):**
  $$x' = \frac{x - \mu}{\sigma}$$
  Đưa phân phối về trung bình bằng 0 và độ lệch chuẩn bằng 1.
* *Quy tắc sống còn:* Luôn `fit_transform` trên tập Train, nhưng **chỉ `transform` trên tập Test**!

### 2. Cấu trúc Báo cáo Kết quả Machine Learning (ML Result Report) chuẩn Olympic
1. **Bài toán & Dữ liệu (Problem Formulation):** Mục tiêu dự đoán, tỷ lệ phân chia Train/Test, kiểm tra mất cân bằng nhãn.
2. **Thiết lập Baseline:** Hiệu năng của Dummy Model (sàn kỹ thuật).
3. **Bảng so sánh mô hình (Model Benchmark Table):** Bảng tổng hợp so sánh các thuật toán (Baseline vs Logistic Regression vs Decision Tree vs Random Forest) trên cả Train và Test qua các metric cốt lõi (Accuracy, F1, Precision, Recall).
4. **Phân tích lỗi (Error Analysis):** Khám phá những trường hợp mô hình đoán sai nhiều nhất và lý giải nguyên nhân dữ liệu.
5. **Kết luận & Bước đi tiếp theo:** Khuyến nghị hướng cải tiến (ví dụ: cần thêm đặc trưng gì, hay cần chuyển sang mạng nơ-ron sâu ở Module M3).



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình End-to-End ML Pipeline:
[Raw Dataset]
      │
[Train / Test Split] (Stratified 80/20)
      │
[Feature Scaler] ──> fit_transform(X_train) ──> transform(X_test)
      │
[Model Arena] ───┬──> Model 1: Baseline (Dummy)
                 ├──> Model 2: Logistic Regression
                 ├──> Model 3: Decision Tree
                 └──> Model 4: Random Forest
      │
[Evaluation & Benchmark Table] (Accuracy, F1-Score, Confusion Matrix)
      │
[Official ML Result Report & Certification]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Khoi tao bai toan thuc te: Du doan kha nang roi bo dich vu (Customer Churn)
X, y = make_classification(
    n_samples=600, n_features=4, n_informative=3, n_redundant=1,
    weights=[0.75, 0.25], flip_y=0.03, random_state=101
)

feature_names = ['Account_Age', 'Monthly_Usage', 'Support_Tickets', 'Bill_Amount']
df_features = pd.DataFrame(X, columns=feature_names)

# Chia tap du lieu chuan hoa
X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.25, random_state=42, stratify=y
)

# Chuan hoa dac trung (Chong Data Leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tap Train: {X_train.shape[0]} mau | Tap Test: {X_test.shape[0]} mau")
print("Ty le khach hang roi bo (Churn=1) trong tap Test: {:.1f}%".format(y_test.mean() * 100))

### 🔍 Code Walkthrough (Khối 1)
Khối code khởi tạo bài toán dự đoán rời bỏ dịch vụ thực tế:
- Bộ dữ liệu gồm 4 đặc trưng liên tục với thang đo khác biệt.
- Sử dụng `StandardScaler` đúng chuẩn chống rò rỉ dữ liệu (`fit_transform` trên Train, `transform` trên Test).


In [ ]:
# 2. Dau truong so sanh hieu nang (Model Benchmark Arena)
models = {
    'Baseline (Majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree (d=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest (50 trees)': RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
}

results = []

for name, model in models.items():
    # Huan luyen tren tap Train da scale
    model.fit(X_train_scaled, y_train)
    
    # Du doan tren ca Train va Test de kiem tra Overfitting
    train_preds = model.predict(X_train_scaled)
    test_preds = model.predict(X_test_scaled)
    
    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    test_prec = precision_score(y_test, test_preds, zero_division=0)
    test_rec = recall_score(y_test, test_preds, zero_division=0)
    test_f1 = f1_score(y_test, test_preds, zero_division=0)
    
    results.append({
        'Model': name,
        'Train Acc': f"{train_acc:.3f}",
        'Test Acc': f"{test_acc:.3f}",
        'Test Precision': f"{test_prec:.3f}",
        'Test Recall': f"{test_rec:.3f}",
        'Test F1-Score': f"{test_f1:.3f}"
    })

df_results = pd.DataFrame(results)
print("===== BANG SO SANH HIEU NANG MO HINH (BENCHMARK TABLE) =====")
print(df_results.to_string(index=False))

### 🔍 Code Walkthrough (Khối 2)
Khối code xây dựng một khung đánh giá tự động đa mô hình:
- Tự động lặp qua 4 mô hình, huấn luyện và đánh giá trên 5 chỉ số khác nhau.
- Xuất ra Bảng Benchmark trực quan thể hiện rõ ràng sự vượt trội của các mô hình học máy so với Baseline ngây thơ.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phân tích bảng kết quả Benchmark
Dựa vào bảng kết quả trên, hãy cho biết:
1. Mô hình nào đạt điểm $F1-Score$ cao nhất trên tập Test?
2. Có hiện tượng Overfitting nghiêm trọng xảy ra ở mô hình nào không (so sánh chênh lệch giữa `Train Acc` và `Test Acc`)?

### Bài 2 (Easy): Thử nghiệm với thuật toán K-Láng giềng gần nhất (KNN)
Import `KNeighborsClassifier` từ `sklearn.neighbors`. Thêm mô hình KNN với `n_neighbors=5` vào dictionary `models` và chạy lại bảng đánh giá. Giải thích tại sao KNN bắt buộc phải chuẩn hóa đặc trưng (Scaling) trước khi chạy.

### Bài 3 (Medium): Xuất báo cáo Markdown tự động
Viết một hàm `export_ml_report(df_results, filename="ML_RESULT_REPORT.md")` tự động ghi bảng so sánh hiệu năng kèm nhận xét vào một file Markdown trong thư mục làm việc của bạn.

### Bài 4 (Medium): Phân tích trường hợp đoán sai (Error Analysis)
Lọc ra các chỉ số index của tập Test mà mô hình tốt nhất đưa ra dự đoán sai (`test_preds != y_test`). Quan sát giá trị các đặc trưng của những mẫu bị đoán sai đó và đưa ra nhận xét: Tại sao mô hình lại bị nhầm lẫn ở những trường hợp này?

### Bài 5 (Challenge): Tối ưu siêu tham số đơn giản (Simple Grid Search)
Viết một vòng lặp thử nghiệm tham số `max_depth` của Decision Tree từ 1 đến 10. Vẽ biểu đồ thể hiện điểm `Train F1` và `Test F1` theo `max_depth` để tìm ra độ sâu tối ưu nhất giúp mô hình tránh cả Underfitting lẫn Overfitting.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Khi nộp bài thi Olympic AI, một báo cáo kỹ thuật rõ ràng với bảng so sánh Baseline vs Candidate Models là minh chứng cho sự chuyên nghiệp của bạn. Ban giám khảo đánh giá rất cao những thí sinh hiểu rõ:
1. Tại sao mình chọn mô hình A mà không chọn mô hình B?
2. Mô hình có bị overfit hay không?
3. Tradeoff giữa tốc độ suy diễn và độ chính xác F1 là bao nhiêu?
Làm chủ kỹ năng lập báo cáo này là bạn đã tiến sát tới tiêu chuẩn của một vận động viên AI cấp quốc gia.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một thí sinh chuẩn hóa dữ liệu bằng `StandardScaler` trước khi chia `train_test_split`:

```python
# CODE CÓ LỖI RÒ RỈ DỮ LIỆU NGHIÊM TRỌNG:
# scaler = StandardScaler()
# X_all_scaled = scaler.fit_transform(X) # Rò rỉ Mean và Std của tập Test vào tập Train!
# X_train, X_test, y_train, y_test = train_test_split(X_all_scaled, y)
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc tính `mean` và `std` trên toàn bộ tập dữ liệu trước khi chia lại vi phạm tính trung thực của kiểm thử (Data Leakage).
2. Viết lại quy trình chuẩn: Phân chia tập trước -> `fit_transform` trên `X_train` -> `transform` trên `X_test`.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Thứ tự đúng đắn của một quy trình Machine Learning chuẩn mực là gì?
   - A. Huấn luyện mô hình -> Thu thập dữ liệu -> Đánh giá -> Chia Train/Test.
   - B. Chia Train/Test -> Chuẩn hóa đặc trưng (Fit trên Train) -> Xây dựng Baseline -> Thử nghiệm mô hình -> Đánh giá đa metric.
   - C. Chuẩn hóa toàn bộ dữ liệu -> Huấn luyện mô hình phức tạp nhất -> Bỏ qua Baseline.
   - D. Đánh giá trên tập Train -> Nộp bài.

2. Tại sao chúng ta chỉ được gọi `scaler.transform(X_test)` mà KHÔNG ĐƯỢC gọi `scaler.fit_transform(X_test)`?
   - A. Vì hàm `fit_transform` không tồn tại trong Scikit-Learn.
   - B. Vì tập Test phải được chuẩn hóa bằng chính các tham số thống kê ($\mu, \sigma$) đã học từ tập Train, nhằm mô phỏng dữ liệu mới đến từ thực tế.
   - C. Vì làm như vậy sẽ gây tràn bộ nhớ RAM.
   - D. Để tiết kiệm thời gian chạy code.

3. Dấu hiệu rõ ràng nhất của hiện tượng Overfitting trên bảng Benchmark là gì?
   - A. Cả điểm Train và điểm Test đều rất thấp.
   - B. Điểm trên tập Train rất cao (gần 100%), nhưng điểm trên tập Test tụt dốc thảm hại.
   - C. Thời gian huấn luyện mô hình quá nhanh.
   - D. Mô hình dự đoán ra toàn số 0.

4. Trong bảng so sánh các mô hình, vai trò của `DummyClassifier` là gì?
   - A. Là mô hình dự thi chính thức để giành huy chương.
   - B. Đóng vai trò là mốc cơ sở tối thiểu (Baseline) để chứng minh các mô hình học máy thực sự có giá trị tri thức.
   - C. Tự động sửa lỗi cú pháp trong code.
   - D. Tăng độ chính xác của Random Forest.

5. Sản phẩm đầu ra chính thức mà học viên hoàn thành khi kết thúc Module M2 là gì?
   - A. Một mạng nơ-ron sâu 100 lớp viết bằng PyTorch.
   - B. Model phân loại hoặc dự đoán đầu tiên kèm Báo cáo kết quả (First ML Model & Result Report).
   - C. Một ứng dụng di động AI.
   - D. Bản đồ học tập cá nhân.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Quy trình khép kín xây dựng một End-to-End Machine Learning Pipeline.
  - Kỹ thuật chuẩn hóa đặc trưng an toàn không rò rỉ dữ liệu (StandardScaler).
  - So sánh đối sánh định lượng giữa Baseline, Logistic Regression, Decision Tree và Random Forest.
  - Cách thực hiện Error Analysis và cấu trúc một Báo cáo Kết quả ML chuẩn mực.
* **What I can now do:**
  - Tự tin huấn luyện mô hình Machine Learning đầu tiên từ bảng dữ liệu thực tế.
  - Phân tích và báo cáo kết quả thực nghiệm đạt chuẩn thi đấu khoa học.
* **What I still need to practice:**
  - Chuẩn bị bước vào thế giới **Deep Learning và PyTorch** ở Module M3!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Xin chúc mừng bạn đã chính thức chinh phục thành công Module M2 và nhận Chứng chỉ **Certificate of Machine Learning Basics**!
Trong buổi tiếp theo (**M3_S01**), chúng ta sẽ bước lên một nấc thang công nghệ mới: **Module M3: XÂY MODEL AI SÂU HƠN BẰNG PYTORCH** — Bắt đầu làm chủ Tensors, Autograd và đồ thị tính toán tự động!




---

## 🏆 MODULE 2 ASSESSMENT & CERTIFICATION

### Phần A: Đánh giá Lý thuyết & Tư duy Mô hình (Theory & Model Reasoning)
- Nắm vững công thức ánh xạ $y = f(X)$, phân biệt Regression vs Classification.
- Giải thích bản chất hàm mất mát (MSE vs Log-Loss) và ranh giới phân tách của Decision Tree.
- Bóc trần bản chất của Accuracy Paradox và giải thích cán cân Precision-Recall.

### Phần B: Sản phẩm Thực hành Bắt buộc (Practical Deliverable)
- **Model phân loại hoặc dự đoán đầu tiên:** Huấn luyện thành công ít nhất 2 mô hình máy học (Linear/Logistic, Tree/Forest) có so sánh với Baseline.
- **Báo cáo kết quả chuẩn thi đấu (ML Result Report):** Bảng Benchmark đa metric (Accuracy, Precision, Recall, F1 hoặc MAE, RMSE) kèm phân tích hiện tượng Overfitting và phân tích lỗi sai (Error Analysis).

### Tiêu chí Đạt chuẩn (Rubric):
| Tiêu chí | Trọng số | Yêu cầu đạt |
| :--- | :--- | :--- |
| **Pipeline & Chống Leakage** | 30% | Phân chia tập chuẩn hóa, scaling đúng quy cách |
| **So sánh Baseline & Models** | 25% | Có mô hình Baseline ngây thơ và đánh bại được Baseline |
| **Đánh giá Đa Metric** | 25% | Đọc đúng Confusion Matrix, Precision, Recall, F1 |
| **Báo cáo Kết quả & Phân tích lỗi** | 20% | Xuất bảng Benchmark và chỉ ra nguyên nhân lỗi sai |

**Chứng chỉ hoàn thành:**
🏅 **Certificate of Machine Learning Basics** — Đủ điều kiện tiến vào Module M3 (PyTorch Deep Learning).


---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M2**:
- Xây dựng End-to-End ML Pipeline hoàn chỉnh ✓
- Model phân loại/dự đoán đầu tiên đánh bại Baseline ✓
- Báo cáo kết quả chuẩn thi đấu (Benchmark table & Error analysis) ✓
- Nhận chứng chỉ Certificate of Machine Learning Basics ✓

* **Session Deliverable:** Model phân loại/dự đoán đầu tiên + Báo cáo kết quả (ML Result Report) hoàn chỉnh
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
